In [1]:
# 1. Imports and data preparation
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

# Load and order the observations chronologically
csv_path = Path("btc_final.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        "btc_final.csv was not found. Put it in the same folder as this notebook."
    )

df = pd.read_csv(csv_path)
df["date"] = pd.to_datetime(df["date"], dayfirst=True)
df = df.sort_values("date").reset_index(drop=True)

# Keep the exact pct_change() return calculation.
# A zero price makes the return at that row or the following row invalid.
valid_btc_price = df["bt_close"] > 0
df["btc_returns"] = df["bt_close"].pct_change()
df.loc[~(valid_btc_price & valid_btc_price.shift(1, fill_value=False)), "btc_returns"] = np.nan

# S&P returns use their own price-validity mask (not the BTC mask).
valid_sp_price = df["sp_close"] > 0
df["sp_returns"] = df["sp_close"].pct_change()
df.loc[~(valid_sp_price & valid_sp_price.shift(1, fill_value=False)), "sp_returns"] = np.nan

# Forward target: information at t predicts BTC return from t to t+1.
df["btc_target"] = df["btc_returns"].shift(-1)

feature_columns = [
    "bt_close", "bt_volume", "bt_day_diff", "bt_close_off_high", "bt_volatility",
    "eur_exch_rate", "jpy_exch_rate", "cny_exch_rate", "google_trends_bitcoin",
    "avg_block_size", "transactions", "difficulty", "bchain_size", "mining_revenue",
    "hash_rate", "cost_per_transaction", "sp_close", "dj_close", "nasdaq_close",
    "vix_close", "gold_am", "silver", "oil_price", "btc_returns", "sp_returns"
]

# Retain dates for charts, while keeping only model features and target in X/y.
model_data = df[["date"] + feature_columns + ["btc_target"]].copy()
model_data = model_data.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

X = model_data[feature_columns]
y = model_data["btc_target"]

# Chronological 80/20 split. No shuffling.
split_index = int(len(model_data) * 0.80)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

train_start = model_data["date"].iloc[0]
train_end = model_data["date"].iloc[split_index - 1]
test_start = model_data["date"].iloc[split_index]
test_end = model_data["date"].iloc[-1]

print("Final dataset:", model_data.shape)
print("Features:", X.shape[1])
print("Training:", X_train.shape, train_start.date(), "to", train_end.date())
print("Testing: ", X_test.shape, test_start.date(), "to", test_end.date())
print("Missing values:", int(model_data.isna().sum().sum()))
print("Infinite values:", int(np.isinf(model_data.select_dtypes(include=np.number)).sum().sum()))
print("\nForward target summary:")
print(y.describe())

Final dataset: (1966, 27)
Features: 25
Training: (1572, 25) 2013-04-29 to 2017-08-22
Testing:  (394, 25) 2017-08-23 to 2018-09-20
Missing values: 0
Infinite values: 0

Forward target summary:
count    1966.000000
mean        0.004632
std         0.065654
min        -0.590909
25%        -0.016207
50%         0.002383
75%         0.023805
max         1.249998
Name: btc_target, dtype: float64


In [2]:
# 2. Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)
linear_pred = linear_model.predict(X_test_scaled)

# 3. Random Forest — same chronological split
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# Naive baseline: predict the training-period mean for every test observation.
baseline_pred = np.full(len(y_test), y_train.mean())

def regression_metrics(actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(actual, predicted),
    }

linear_scores = regression_metrics(y_test, linear_pred)
rf_scores = regression_metrics(y_test, rf_pred)
baseline_scores = regression_metrics(y_test, baseline_pred)

# Additional ranking/direction diagnostics for Random Forest.
rf_directional_accuracy = np.mean((rf_pred > 0) == (y_test.to_numpy() > 0))
rf_ic, rf_ic_pvalue = spearmanr(rf_pred, y_test.to_numpy())

score_table = pd.DataFrame(
    [linear_scores, rf_scores, baseline_scores],
    index=["Linear Regression", "Random Forest", "Training-mean baseline"]
)

print("TEST-SET RESULTS")
display(score_table)
print(f"RF directional accuracy: {rf_directional_accuracy:.4%}")
print(f"RF Spearman IC: {rf_ic:.6f}")
print(f"RF IC p-value: {rf_ic_pvalue:.6f}")

print("\nFirst 10 Random Forest predictions:")
print(np.round(rf_pred[:10], 6))
print("First 10 actual next-day returns:")
print(np.round(y_test.iloc[:10].to_numpy(), 6))

TEST-SET RESULTS


,MAE,MSE,RMSE,R2
Linear Regression,0.355044,0.212901,0.461412,-82.126934
Random Forest,0.036835,0.002621,0.051199,-0.023507
Training-mean baseline,0.035775,0.002568,0.050677,-0.002728


RF directional accuracy: 54.3147%
RF Spearman IC: 0.070754
RF IC p-value: 0.160999

First 10 Random Forest predictions:
[0.010217 0.011823 0.010044 0.012176 0.01232  0.010909 0.010703 0.010546
 0.013661 0.017787]
First 10 actual next-day returns:
[ 0.04482   0.008673 -0.004416  0.007003  0.        0.045665 -0.003104
  0.030687  0.040777 -0.065091]


In [3]:
# 4. Compact diagnostic summary for the experiment
rmse_improvement_vs_linear = 1 - rf_scores["RMSE"] / linear_scores["RMSE"]
rf_beats_baseline_rmse = rf_scores["RMSE"] < baseline_scores["RMSE"]
rf_beats_baseline_mae = rf_scores["MAE"] < baseline_scores["MAE"]

print(f"RF RMSE reduction vs Linear Regression: {rmse_improvement_vs_linear:.2%}")
print(f"RF beats training-mean baseline on RMSE: {rf_beats_baseline_rmse}")
print(f"RF beats training-mean baseline on MAE:  {rf_beats_baseline_mae}")
print(f"Test-period dates: {test_start.date()} to {test_end.date()}")

# Feature importance is a model diagnostic, not proof of causality.
feature_importance = pd.Series(
    rf_model.feature_importances_, index=feature_columns
).sort_values(ascending=False)
display(feature_importance.head(12).to_frame("RF impurity importance"))

RF RMSE reduction vs Linear Regression: 88.90%
RF beats training-mean baseline on RMSE: False
RF beats training-mean baseline on MAE:  False
Test-period dates: 2017-08-23 to 2018-09-20


,RF impurity importance
sp_close,0.179067
avg_block_size,0.170925
bt_close,0.111265
btc_returns,0.088423
mining_revenue,0.069744
bt_volatility,0.050850
bt_day_diff,0.037409
jpy_exch_rate,0.033320
cost_per_transaction,0.032290
nasdaq_close,0.023144


In [4]:
# Register Manim's notebook magic once in this kernel.
from manim import *
from manim.utils.ipython_magic import ManimMagic
get_ipython().register_magics(ManimMagic)

print("Manim is ready. Run the next cell to render the explainer.")

Manim is ready. Run the next cell to render the explainer.


In [5]:
%%manim -qm -v WARNING BTCModelLab

class BTCModelLab(Scene):
    def txt(self, text, size=24, color=WHITE, weight=None):
        kwargs = {} if weight is None else {"weight": weight}
        return Text(text, font_size=size, color=color, **kwargs)

    def clear_scene(self):
        if self.mobjects:
            self.play(*[FadeOut(m) for m in list(self.mobjects)], run_time=0.45)

    def header(self, title, subtitle=None, color=BLUE_B):
        h = self.txt(title, 34, color, BOLD).to_edge(UP)
        self.play(Write(h), run_time=0.7)
        if subtitle:
            s = self.txt(subtitle, 19, GREY_B).next_to(h, DOWN, buff=0.15)
            self.play(FadeIn(s), run_time=0.4)
        return h

    def metric_line(self, label, value, color=WHITE):
        a = self.txt(label, 22, GREY_A)
        b = self.txt(value, 23, color, BOLD)
        return VGroup(a, b).arrange(RIGHT, buff=0.35, aligned_edge=DOWN)

    def card(self, title, value, color=BLUE_B, width=2.7):
        rect = RoundedRectangle(
            width=width, height=1.15, corner_radius=0.12,
            stroke_color=color, stroke_width=2, fill_color=color, fill_opacity=0.10
        )
        t = self.txt(title, 17, GREY_A).move_to(rect.get_center() + UP*0.22)
        v = self.txt(value, 25, color, BOLD).move_to(rect.get_center() + DOWN*0.22)
        return VGroup(rect, t, v)

    def construct(self):
        # Pull the exact values from the model-training cell.
        lm, rm, bm = linear_scores, rf_scores, baseline_scores
        dates = model_data["date"]
        n_train, n_test = len(X_train), len(X_test)

        # 1 — Opening
        title = self.txt("BTC QUANT MODEL LAB", 43, BLUE_B, BOLD)
        sub = self.txt("A visual walkthrough: Linear Regression → Random Forest", 22, GREY_A)
        sub.next_to(title, DOWN, buff=0.3)
        pulse = Circle(radius=0.55, color=GREEN, stroke_width=5).next_to(title, UP, buff=0.45)
        pulse_dot = Dot(color=GREEN).move_to(pulse.get_center())
        self.play(FadeIn(pulse), FadeIn(pulse_dot), Write(title), run_time=1.2)
        self.play(FadeIn(sub))
        self.play(pulse.animate.scale(1.25), pulse_dot.animate.scale(1.3), rate_func=there_and_back)
        self.wait(1)
        self.clear_scene()

        # 2 — Data pipeline
        self.header("01 / MARKET DATA → PREDICTION", "25 features • daily observations • forward one-day target")
        labels = ["BTC price & volume", "On-chain activity", "Macro markets", "Daily returns"]
        colors = [BLUE_B, TEAL, PURPLE, GOLD]
        boxes = VGroup()
        for lab, col in zip(labels, colors):
            r = RoundedRectangle(width=2.45, height=0.85, corner_radius=0.12,
                                 stroke_color=col, fill_color=col, fill_opacity=0.12)
            t = self.txt(lab, 17, col).move_to(r)
            boxes.add(VGroup(r, t))
        boxes.arrange(RIGHT, buff=0.22).shift(UP*0.65)
        self.play(LaggedStart(*[FadeIn(b, shift=UP*0.2) for b in boxes], lag_ratio=0.15))
        arrow = Arrow(boxes.get_bottom()+DOWN*0.18, DOWN*0.75, color=GREY_B)
        target = self.card("PREDICTION TARGET", "BTC return t+1", GREEN, 3.4).shift(DOWN*1.55)
        formula = self.txt("X(t)  ───────────────▶  rBTC(t+1)", 27, YELLOW, BOLD).move_to(UP*0.05)
        self.play(GrowArrow(arrow), Write(formula), FadeIn(target))
        self.wait(1.8)
        self.clear_scene()

        # 3 — Chronological split
        self.header("02 / TIME-AWARE VALIDATION", "Never shuffle financial time series")
        total_w = 10.8
        train_w = total_w * n_train / (n_train+n_test)
        test_w = total_w - train_w
        tr = Rectangle(width=train_w, height=0.9, stroke_color=BLUE_B,
                       fill_color=BLUE_B, fill_opacity=0.65)
        te = Rectangle(width=test_w, height=0.9, stroke_color=GREEN,
                       fill_color=GREEN, fill_opacity=0.7)
        tr.move_to(LEFT*(test_w/2)+LEFT*0.05)
        te.next_to(tr, RIGHT, buff=0)
        tr_lab = self.txt(f"TRAIN  {n_train:,}", 23, WHITE, BOLD).move_to(tr)
        te_lab = self.txt(f"TEST  {n_test:,}", 18, WHITE, BOLD).move_to(te)
        dates1 = self.txt(f"{train_start:%d %b %Y}  →  {train_end:%d %b %Y}", 18, BLUE_B)
        dates2 = self.txt(f"{test_start:%d %b %Y}  →  {test_end:%d %b %Y}", 17, GREEN)
        dates1.next_to(tr, DOWN, buff=0.35)
        dates2.next_to(te, DOWN, buff=0.35)
        note = self.txt("Fit on the past. Evaluate on unseen future observations.", 21, YELLOW).to_edge(DOWN)
        self.play(Create(tr), Create(te), FadeIn(tr_lab), FadeIn(te_lab))
        self.play(FadeIn(dates1), FadeIn(dates2), Write(note))
        self.wait(2)
        self.clear_scene()

        # 4 — Linear regression model architecture
        self.header("03 / MODEL ONE — LINEAR REGRESSION", "One weighted equation maps the features to a return estimate", BLUE_B)
        eq = self.txt("prediction = intercept + (weight × feature) + ...", 26, YELLOW, BOLD)
        eq.move_to(UP*1.0)
        feature_nodes = VGroup(*[
            Circle(radius=0.23, color=BLUE_B, fill_opacity=0.35).move_to(LEFT*4.3 + RIGHT*i*1.05 + DOWN*0.45)
            for i in range(5)
        ])
        node_labels = VGroup(*[
            self.txt(s, 14, GREY_A).next_to(node, DOWN, buff=0.18)
            for s, node in zip(["price", "volume", "on-chain", "macro", "returns"], feature_nodes)
        ])
        output = self.card("MODEL OUTPUT", "ŷ(t+1)", GOLD, 2.4).move_to(RIGHT*3.8 + DOWN*0.45)
        lines = VGroup(*[
            Line(node.get_right(), output.get_left(), color=GREY_B, stroke_opacity=0.55)
            for node in feature_nodes
        ])
        self.play(Write(eq), FadeIn(feature_nodes), FadeIn(node_labels))
        self.play(Create(lines), FadeIn(output))
        moving = Dot(color=YELLOW, radius=0.09).move_to(feature_nodes[0].get_center())
        self.play(FadeIn(moving))
        self.play(MoveAlongPath(moving, Line(feature_nodes[0].get_center(), output.get_center())), run_time=1.0)
        self.wait(1.3)
        self.clear_scene()

        # 5 — Linear Regression scores, generated from corrected forward target
        self.header("LINEAR REGRESSION / TEST RESULTS", "Actual recomputed scores for the forward next-day target", BLUE_B)
        rows = VGroup(
            self.metric_line("MAE", f"{lm['MAE']:.6f}", RED),
            self.metric_line("MSE", f"{lm['MSE']:.6f}", RED),
            self.metric_line("RMSE", f"{lm['RMSE']:.6f}", RED),
            self.metric_line("R²", f"{lm['R2']:.6f}", RED),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.32).move_to(UP*0.1)
        self.play(LaggedStart(*[FadeIn(r, shift=RIGHT*0.2) for r in rows], lag_ratio=0.18))
        caution = self.txt("Large errors: this linear fit struggles to generalise to the test period.", 19, RED).to_edge(DOWN)
        self.play(Write(caution))
        self.wait(2)
        self.clear_scene()

        # 6 — Random Forest architecture: three representative tree diagrams
        self.header("04 / MODEL TWO — RANDOM FOREST", "500 decision trees learn different feature splits", GREEN)
        tree_groups = VGroup()
        tree_centers = [-3.5, 0, 3.5]
        for cx in tree_centers:
            root = Dot([cx, 1.05, 0], radius=0.11, color=YELLOW)
            mids = [Dot([cx-0.65, 0.0, 0], radius=0.095, color=BLUE_B),
                    Dot([cx+0.65, 0.0, 0], radius=0.095, color=BLUE_B)]
            leaves = [Dot([cx-0.98, -0.95, 0], radius=0.075, color=GREEN),
                      Dot([cx-0.32, -0.95, 0], radius=0.075, color=GREEN),
                      Dot([cx+0.32, -0.95, 0], radius=0.075, color=GREEN),
                      Dot([cx+0.98, -0.95, 0], radius=0.075, color=GREEN)]
            segs = VGroup(
                Line(root.get_center(), mids[0].get_center(), color=GREY_B),
                Line(root.get_center(), mids[1].get_center(), color=GREY_B),
                Line(mids[0].get_center(), leaves[0].get_center(), color=GREY_B),
                Line(mids[0].get_center(), leaves[1].get_center(), color=GREY_B),
                Line(mids[1].get_center(), leaves[2].get_center(), color=GREY_B),
                Line(mids[1].get_center(), leaves[3].get_center(), color=GREY_B),
            )
            tree_groups.add(VGroup(segs, root, *mids, *leaves))
        self.play(LaggedStart(*[Create(t) for t in tree_groups], lag_ratio=0.2))
        tree_note = self.txt("3 representative trees shown — the fitted forest contains 500.", 18, GREY_A).to_edge(DOWN)
        self.play(FadeIn(tree_note))
        self.wait(1.5)
        self.clear_scene()

        # 7 — Ensemble aggregation
        self.header("HOW THE FOREST MAKES ONE PREDICTION", "Each tree votes with a numeric return estimate", GREEN)
        tree_outputs = VGroup(*[
            self.card(f"TREE {i}", "estimate", GREEN, 2.0)
            for i in range(1, 4)
        ]).arrange(RIGHT, buff=0.35).shift(UP*0.65)
        arrows = VGroup(*[
            Arrow(c.get_bottom(), DOWN*0.65, buff=0.1, color=GREY_B)
            for c in tree_outputs
        ])
        avg = self.card("AGGREGATION", "AVERAGE", YELLOW, 3.0).shift(DOWN*1.2)
        formula = self.txt("RF prediction = mean(tree 1, tree 2, ... tree 500)", 22, YELLOW)
        formula.to_edge(DOWN)
        self.play(LaggedStart(*[FadeIn(c) for c in tree_outputs], lag_ratio=0.15))
        self.play(*[GrowArrow(a) for a in arrows])
        self.play(FadeIn(avg), Write(formula))
        self.wait(2)
        self.clear_scene()

        # 8 — Random Forest scores
        self.header("RANDOM FOREST / TEST RESULTS", "Same forward target and same chronological test window", GREEN)
        rows = VGroup(
            self.metric_line("MAE", f"{rm['MAE']:.6f}", GREEN),
            self.metric_line("MSE", f"{rm['MSE']:.6f}", GREEN),
            self.metric_line("RMSE", f"{rm['RMSE']:.6f}", GREEN),
            self.metric_line("R²", f"{rm['R2']:.6f}", YELLOW),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.30).move_to(UP*0.1)
        self.play(LaggedStart(*[FadeIn(r, shift=RIGHT*0.2) for r in rows], lag_ratio=0.18))
        note = self.txt("Lower errors than Linear Regression — but compare against a naive baseline.", 18, YELLOW).to_edge(DOWN)
        self.play(Write(note))
        self.wait(2)
        self.clear_scene()

        # 9 — Real test observations chart
        self.header("ACTUAL VS PREDICTED / FIRST 30 TEST DAYS", "Returns are shown as decimals; chart values come directly from the test set", BLUE_B)
        n = min(30, len(y_test))
        actual_sample = y_test.iloc[:n].to_numpy()
        pred_sample = np.asarray(rf_pred[:n])
        ymin = float(min(actual_sample.min(), pred_sample.min()))
        ymax = float(max(actual_sample.max(), pred_sample.max()))
        pad = max((ymax-ymin)*0.12, 0.005)
        ax = Axes(
            x_range=[0, n-1, 5], y_range=[ymin-pad, ymax+pad, max((ymax-ymin+2*pad)/4, 0.01)],
            x_length=10.2, y_length=4.1,
            axis_config={"color": GREY_B, "include_numbers": False}
        ).shift(DOWN*0.45)
        actual_line = ax.plot_line_graph(
            x_values=list(range(n)), y_values=actual_sample.tolist(),
            line_color=BLUE_B, add_vertex_dots=False
        )
        pred_line = ax.plot_line_graph(
            x_values=list(range(n)), y_values=pred_sample.tolist(),
            line_color=YELLOW, add_vertex_dots=False
        )
        legend1 = self.txt("Actual next-day return", 17, BLUE_B)
        legend2 = self.txt("RF prediction", 17, YELLOW)
        legend = VGroup(legend1, legend2).arrange(RIGHT, buff=0.55).next_to(ax, UP, buff=0.12)
        self.play(Create(ax), FadeIn(legend))
        self.play(Create(actual_line), run_time=1.5)
        self.play(Create(pred_line), run_time=1.5)
        self.wait(2)
        self.clear_scene()

        # 10 — Error metric comparison bars
        self.header("MODEL COMPARISON / ERROR METRICS", "Shorter bars indicate smaller prediction errors", BLUE_B)
        metric_specs = [
            ("MAE", lm["MAE"], rm["MAE"], bm["MAE"]),
            ("RMSE", lm["RMSE"], rm["RMSE"], bm["RMSE"]),
        ]
        groups = VGroup()
        maxval = max(v for row in metric_specs for v in row[1:])
        for j, (name, lv, rv, bv) in enumerate(metric_specs):
            y = 1.15 - j*2.25
            label = self.txt(name, 22, WHITE, BOLD).move_to(LEFT*5.2 + UP*y)
            bars = VGroup()
            for k, (lab, val, col) in enumerate([
                ("Linear", lv, RED), ("RF", rv, GREEN), ("Mean baseline", bv, BLUE_B)
            ]):
                width = max(0.08, 5.4*val/maxval)
                rect = Rectangle(width=width, height=0.30, stroke_width=0,
                                 fill_color=col, fill_opacity=0.85)
                rect.move_to(LEFT*1.7 + UP*(y-0.42-k*0.42), aligned_edge=LEFT)
                txt = self.txt(f"{lab}: {val:.4f}", 16, col).next_to(rect, RIGHT, buff=0.15)
                bars.add(VGroup(rect, txt))
            groups.add(VGroup(label, bars))
        self.play(LaggedStart(*[FadeIn(g) for g in groups], lag_ratio=0.2))
        baseline_note = self.txt(
            f"RF beats mean baseline on RMSE: {rf_beats_baseline_rmse}  |  MAE: {rf_beats_baseline_mae}",
            18, YELLOW
        ).to_edge(DOWN)
        self.play(Write(baseline_note))
        self.wait(2.2)
        self.clear_scene()

        # 11 — Direction and rank IC
        self.header("SIGNAL QUALITY INDICATORS", "Direction and ranking are different questions", PURPLE)
        dcard = self.card("DIRECTIONAL ACCURACY", f"{rf_directional_accuracy:.2%}", GREEN, 3.4).shift(LEFT*2.0 + UP*0.55)
        iccard = self.card("SPEARMAN IC", f"{rf_ic:.4f}", BLUE_B, 3.0).shift(RIGHT*2.0 + UP*0.55)
        ptxt = self.txt(f"IC p-value = {rf_ic_pvalue:.4f}", 25, YELLOW, BOLD).move_to(DOWN*0.85)
        interpretation = self.txt(
            "Directional accuracy: fraction of up/down calls that were correct.", 18, GREY_A
        ).move_to(DOWN*1.55)
        ranktext = self.txt(
            "IC: association between predicted-return ranks and realised-return ranks.", 18, GREY_A
        ).next_to(interpretation, DOWN, buff=0.2)
        self.play(FadeIn(dcard), FadeIn(iccard), Write(ptxt))
        self.play(FadeIn(interpretation), FadeIn(ranktext))
        pnote = self.txt("A high/low p-value is not a profitability test.", 18, RED).to_edge(DOWN)
        self.play(Write(pnote))
        self.wait(2.5)
        self.clear_scene()

        # 12 — Conclusion
        self.header("RESEARCH VERDICT", "What the corrected forward test currently tells us", YELLOW)
        verdicts = [
            ("Linear Regression", "Weak out-of-sample fit", RED),
            ("Random Forest", "Lower error than Linear Regression", GREEN),
            ("Naive baseline", "RF does not beat it on MAE/RMSE here", YELLOW),
            ("Rank signal", "IC is small; p-value is not significant", YELLOW),
        ]
        items = VGroup()
        for name, desc, col in verdicts:
            left = self.txt(name, 20, col, BOLD)
            right = self.txt(desc, 19, GREY_A)
            items.add(VGroup(left, right).arrange(RIGHT, buff=0.35))
        items.arrange(DOWN, aligned_edge=LEFT, buff=0.38).move_to(UP*0.15)
        self.play(LaggedStart(*[FadeIn(it, shift=RIGHT*0.2) for it in items], lag_ratio=0.2))
        final = self.txt("NEXT: validation design → feature audit → XGBoost → strategy backtest", 21, BLUE_B, BOLD).to_edge(DOWN)
        self.play(Write(final))
        self.wait(3)


Manim Community v0.22.1